# fedOPD data fetch

Three datasets, each in its own folder under `../data`:

| # | dataset | folder |
|---|---------|--------|
| 1 | NHS RTT monthly Full CSV zips, Apr 2018 – Mar 2026 (96 files, ~8 GB extracted) | `data/rtt_zips` -> `data/rtt_csv` |
| 2 | Trust -> ICB / geography reference (ODS ORD API) | `data/reference` |
| 3 | KH03 overnight bed availability & occupancy | `data/kh03` |



In [ ]:
import re
import zipfile
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from bs4 import BeautifulSoup

# folder layout: everything lands under ../data
DATA = Path("../data")
RTT_ZIPS = DATA / "rtt_zips"   # raw monthly zips as downloaded
RTT_CSV = DATA / "rtt_csv"     # extracted csvs (~8 GB)
REF = DATA / "reference"       # trust -> ICB lookup table
KH03 = DATA / "kh03"           # quarterly bed spreadsheets
for d in (RTT_ZIPS, RTT_CSV, REF, KH03):
    d.mkdir(parents=True, exist_ok=True)

# nhs.uk returns 403 to the default python-requests user agent, so pretend to be a browser
HEADERS = {"User-Agent": "Mozilla/5.0"}

## 1. Build the RTT Full CSV manifest

Each financial year (2018-19 ... 2025-26) has one page listing that year's monthly files. We scrape all eight pages, keep only links whose filename says Full-CSV and ends in .zip, and key them by the month token in the filename (e.g. `Mar19`). When a month appears twice (original + revised), the revised link wins.

Cross-checked: this filter picks exactly the official links, e.g. `Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip` and `Full-CSV-data-file-Mar26-ZIP-3M-revised.zip`. Ignore the size hint in the filename — the Mar22 revised file says '111805K' but is actually ~4.5 MB on the server; sizes vary wildly month to month and that's fine.

In [ ]:
# '2018-19', '2019-20', ..., '2025-26' — matches the url slug of each year page
YEARS = [f"{y}-{str(y + 1)[2:]}" for y in range(2018, 2026)]

manifest = {}  # month key like 'Mar19' -> download url
for yr in YEARS:
    page = f"https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/rtt-data-{yr}/"
    soup = BeautifulSoup(requests.get(page, headers=HEADERS).text, "html.parser")
    for a in soup.find_all("a", href=True):           # every link on the page
        href = a["href"]
        name = href.split("/")[-1]                    # just the filename part of the url
        if "full-csv" not in name.lower() or not name.endswith(".zip"):
            continue                                  # skip the XLS views, commissioner files, etc.
        # filenames look like 'Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip'
        # -> pull out the 'Mar19' month token to use as the key
        key = re.search(r"([A-Z][a-z]{2}\d{2})", name).group(1)
        # first sighting of a month goes in; a 'revised' filename overwrites the original
        if key not in manifest or "revised" in name.lower():
            manifest[key] = href

# sort chronologically ('Apr18' ... 'Mar26') by parsing the month token as a date
months = sorted(manifest, key=lambda k: datetime.strptime(k, "%b%y"))
print(len(months), "months:", months[0], "->", months[-1])  # expect 96: Apr18 -> Mar26

## 2. Download all 96 zips

Streamed in 1 MB chunks so big files don't sit in memory. Each file downloads to a `.part` name and is renamed only once fully written — so a completed filename on disk always means a complete download, and rerunning the cell skips finished files and resumes the rest. File sizes vary a lot month to month; that's expected.

In [ ]:
for k in months:
    url = manifest[k]
    dest = RTT_ZIPS / url.split("/")[-1]
    if dest.exists():
        continue                                   # already fully downloaded on a previous run
    tmp = dest.with_suffix(".part")                # write in progress under a temp name
    r = requests.get(url, headers=HEADERS, stream=True)
    with open(tmp, "wb") as f:
        for chunk in r.iter_content(1 << 20):      # 1 MB at a time
            f.write(chunk)
    tmp.rename(dest)                               # rename = download definitely complete
    print(k, dest.name, f"{dest.stat().st_size / 1e6:.1f} MB")

print("done,", len(list(RTT_ZIPS.glob('*.zip'))), "zips on disk")

## 3. Extract every zip into one csv folder

In [ ]:
# each zip holds that month's full csv; dump them all into data/rtt_csv
for z in sorted(RTT_ZIPS.glob("*.zip")):
    zipfile.ZipFile(z).extractall(RTT_CSV)

print(len(list(RTT_CSV.glob('*.csv'))), "csvs extracted")

## 4. Trust -> ICB reference (ODS ORD API)

Two steps: one call lists every organisation with primary role RO197 (= NHS Trust), then one call per trust fetches its full record, from which we take the active RE5 ('is commissioned by') relationship — its target is the trust's ICB, which carries primary role **RO261** in ODS. Verified live: e.g. North London NHS FT -> `Z9B2Z` (NHS West and North London ICB).

The list returns ~550 trusts of which ~250 are Active; we keep the inactive ones too since they appear in the 2018-era RTT months (their `icb_code` stays empty). ~550 small requests, a couple of minutes total.

In [ ]:
ORD = "https://directory.spineservices.nhs.uk/ORD/2-0-0"

# all NHS trusts, current and historic, in one page (~550, under the 1000 limit)
trusts = requests.get(
    f"{ORD}/organisations",
    params={"PrimaryRoleId": "RO197", "Limit": 1000, "_format": "json"},
    headers=HEADERS,
).json()["Organisations"]
print(len(trusts), "trusts")

In [ ]:
rows = []
for t in trusts:
    code = t["OrgId"]                              # provider code, same as in the RTT csvs
    # full org record: relationships, geography, dates
    rec = requests.get(f"{ORD}/organisations/{code}", headers=HEADERS).json()["Organisation"]
    icb_code = ""
    for rel in rec.get("Rels", {}).get("Rel", []):  # walk the trust's relationships
        tgt = rel["Target"]
        # active RE5 = 'is commissioned by'; its target with primary role RO261 is the ICB
        if rel["Status"] == "Active" and rel["id"] == "RE5" and tgt["PrimaryRoleId"]["id"] == "RO261":
            icb_code = tgt["OrgId"]["extension"]
    rows.append({
        "trust_code": code,
        "trust_name": t["Name"],
        "status": t["Status"],                     # Active / Inactive (old trusts merged away)
        "icb_code": icb_code,                      # empty for dissolved trusts — fine
        "postcode": rec["GeoLoc"]["Location"].get("PostCode", ""),
    })

ref = pd.DataFrame(rows)
ref.to_csv(REF / "trust_icb.csv", index=False)
ref.head()

## 5. KH03 overnight beds 

The bed-availability page links every quarterly overnight spreadsheet back to 2010 — 61 files as of Jul 2026, latest `Beds-Open-Overnight-Web_File-Q4-2025-26.xlsx` (verified live). We take any xls/xlsx link with 'night' in the filename and download whatever isn't already on disk.

In [ ]:
page = "https://www.england.nhs.uk/statistics/statistical-work-areas/bed-availability-and-occupancy/bed-data-overnight/"
soup = BeautifulSoup(requests.get(page, headers=HEADERS).text, "html.parser")

# set comprehension dedupes; filenames look like 'Beds-Open-Overnight-Q1-2023-24.xlsx'
links = sorted({
    a["href"] for a in soup.find_all("a", href=True)
    if a["href"].split("/")[-1].lower().endswith((".xls", ".xlsx"))
    and "night" in a["href"].split("/")[-1].lower()
})
print(len(links), "files")

for url in links:
    dest = KH03 / url.split("/")[-1]
    if dest.exists():
        continue                                   # already downloaded
    dest.write_bytes(requests.get(url, headers=HEADERS).content)  # small files, no streaming needed
    print(dest.name)